# Agoda booking analysis

A reproducible exploration of booking volume, average daily rate (ADR), and booking lead time across five cities. One row represents one booking.

**The original Excel workbook is not included.** This notebook defaults to explicitly synthetic demo data. Demo charts are not Agoda findings. To analyze the original data, set `DEMO = False` and provide the workbook path below. Run all cells in order from the repository directory.

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display
from analysis import demo_data, load_workbook, prepare_data, summarize, make_plots

DEMO = True
WORKBOOK = Path('Case_Study_Urgency_Message_Data.xlsx')
LABEL = 'SYNTHETIC DEMO' if DEMO else 'Workbook data'
print(LABEL)

## Load and validate

The loader validates all five city sheets and normalizes the source column typo `accommadation_type_name`. Invalid dates, nonpositive or nonfinite rates, missing required values, invalid star ratings, and impossible stays are excluded and counted. Dates are normalized to calendar days before computing lead time. No rows are deduplicated without a documented booking identifier.

In [ ]:
raw = demo_data() if DEMO else load_workbook(WORKBOOK)
df, audit = prepare_data(raw)
display(audit)
display(df.head())

## Booking patterns and segments

Counts use rows rather than an optional `#` column. Months retain the year, and lead-time intervals are nonoverlapping: same day, 1–2, 3–4, 5–14, 15–29, and 30+ days. Missing segment means remain missing instead of becoming zero-dollar rates.

In [ ]:
tables = summarize(df)
for name, table in tables.items():
    print(name)
    display(table.round(2))

In [ ]:
figures = make_plots(tables, LABEL)
for figure in figures.values():
    display(figure)
    plt.close(figure)

## Correlation and interpretation

Correlation between daily booking volume and booked ADR is descriptive; it does not establish that price causes demand. Booked rates across different hotels and stay dates do not show how the price of a specific room changes over time.

In [ ]:
daily = tables['daily']
correlation = daily['bookings'].corr(daily['average_adr'])
print(f'Daily booking count / mean ADR correlation ({LABEL}): {correlation:.3f}')

## Messaging hypotheses

Use observed lead-time and segment patterns to propose experiments. Claims about remaining rooms, discounts, visitor counts, or deadlines require actual inventory, pricing, and event data. This workbook alone cannot substantiate those claims or estimate conversion uplift.

Compare truthful messages against a control using an A/B test, measuring booking conversion and cancellations. Synthetic demo output only verifies that the pipeline executes. The original slide deck and screenshots remain historical artifacts; rerun with the original workbook before updating their conclusions.

## Export

For CSV summaries, PNG charts, and a data quality report, run `python analysis.py --demo` or `python analysis.py --input PATH_TO_WORKBOOK`. Outputs are separated into demo and workbook directories.